# Module 1: Data Acquisition

**Objective**: Acquire all datasets using the hybrid GEE strategy.

- **Tier 1**: Server-side computation at 10m → export CSVs/GeoJSONs
- **Tier 2**: Full-state LULC rasters at 30m (compressed) for visualization
- **Tier 3**: Targeted 10m crops (after Module 2 identifies highest-change district)

**Data Sources**:
- LULC: Google Dynamic World V1
- DEM: SRTM 30m
- Boundaries: FAO GAUL Level 2
- Roads: OpenStreetMap via osmnx

In [ ]:
import sys
sys.path.insert(0, '..')

import ee
import json
import pandas as pd
import geopandas as gpd
from pathlib import Path

from config import (
    STATES, YEARS, ALL_YEARS, DW_ASSET, SRTM_ASSET,
    GAUL_LEVEL1_ASSET, GAUL_LEVEL2_ASSET,
    TIER1_SCALE, TIER2_SCALE, TIER3_SCALE,
    GEE_DRIVE_FOLDER, GEE_CRS, GEE_MAX_PIXELS,
    RAW_DIR, GEE_EXPORTS_DIR, BOUNDARIES_DIR, ROADS_DIR, DEM_DIR, LULC_DIR,
    MAJOR_ROAD_TYPES,
)
from scripts.gee_export import (
    authenticate_gee, get_state_boundary, make_annual_composite,
    compute_state_composition, compute_district_composition,
    compute_transition_matrix_state, compute_transition_matrix_districts,
    vectorize_water_bodies,
    export_raster_to_drive, export_feature_collection_to_drive,
    monitor_tasks, get_srtm_dem, fc_to_dataframe,
)

print('Imports successful!')

## Step 1: Authenticate & Initialize GEE

You need a GEE-enabled Google account. This will open a browser for authentication.

In [ ]:
# Replace with your GEE project ID
GEE_PROJECT_ID = 'your-project-id'  # <-- UPDATE THIS

authenticate_gee(project_id=GEE_PROJECT_ID)

## Step 2: Define Regions of Interest (ROIs)

Load state and district boundaries from FAO GAUL dataset.

In [ ]:
# Load boundaries for both states
boundaries = {}

for state_key, state_cfg in STATES.items():
    gaul_name = state_cfg['gaul_name']
    
    # State boundary (level 1)
    state_fc = get_state_boundary(gaul_name, level='level1')
    
    # District boundaries (level 2)
    district_fc = get_state_boundary(gaul_name, level='level2')
    
    boundaries[state_key] = {
        'state_fc': state_fc,
        'district_fc': district_fc,
    }
    
    # Print district count to verify
    n_districts = district_fc.size().getInfo()
    print(f'{state_cfg["name"]}: {n_districts} districts found')

## Step 3: Create Annual LULC Composites

For each state and year, create a mode composite from Dynamic World V1.

In [ ]:
# Create composites for all state-year combinations
composites = {}

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    composites[state_key] = {}
    
    for year in YEARS:
        print(f'Creating composite: {state_cfg["name"]} {year}...')
        composite = make_annual_composite(year, state_fc)
        composites[state_key][year] = composite
    
    print(f'  ✅ {state_cfg["name"]}: {len(YEARS)} composites ready')

print('\n✅ All composites created (server-side, no download yet).')

## Step 4: TIER 1 — Server-Side Computation

Compute all zonal statistics at 10m native resolution inside GEE.
Only the results (small CSVs) will be exported — NOT the full rasters.

### 4.1: State-Level Composition

In [ ]:
# State-level LULC composition (pixel counts per class)
state_compositions = {}

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    state_compositions[state_key] = {}
    
    for year in YEARS:
        print(f'Computing state composition: {state_cfg["name"]} {year}...')
        result = compute_state_composition(
            composites[state_key][year], state_fc, scale=TIER1_SCALE
        )
        state_compositions[state_key][year] = result.get('label', {})
        print(f'  → {len(state_compositions[state_key][year])} classes found')

# Save to JSON
GEE_EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
with open(GEE_EXPORTS_DIR / 'state_compositions.json', 'w') as f:
    json.dump(state_compositions, f, indent=2)
print('\n💾 Saved: state_compositions.json')

### 4.2: District-Level Composition

In [ ]:
# District-level LULC composition
# NOTE: This can be slow due to many districts. 
# For Maharashtra (36 districts × 3 years = 108 reduce operations)

district_compositions = {}

for state_key, state_cfg in STATES.items():
    district_fc = boundaries[state_key]['district_fc']
    district_compositions[state_key] = {}
    
    for year in YEARS:
        print(f'Computing district composition: {state_cfg["name"]} {year}...')
        result_fc = compute_district_composition(
            composites[state_key][year], district_fc, scale=TIER1_SCALE
        )
        
        # Export to Drive as CSV (large FeatureCollections can't be fetched directly)
        export_feature_collection_to_drive(
            result_fc,
            description=f'{state_key}_district_composition_{year}',
            folder=GEE_DRIVE_FOLDER,
            file_format='CSV'
        )
        
        district_compositions[state_key][year] = result_fc
        print(f'  → Export queued')

print('\n✅ All district composition exports queued.')
print('   Download CSV files from Google Drive after they complete.')

### 4.3: State-Level Transition Matrices

In [ ]:
# Transition matrices for period pairs
period_pairs = [
    (2016, 2020),
    (2020, 2025),
    (2016, 2025),
]

state_transitions = {}

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    state_transitions[state_key] = {}
    
    for year_t1, year_t2 in period_pairs:
        label = f'{year_t1}_{year_t2}'
        print(f'Computing transition: {state_cfg["name"]} {year_t1}→{year_t2}...')
        
        result = compute_transition_matrix_state(
            composites[state_key][year_t1],
            composites[state_key][year_t2],
            state_fc, scale=TIER1_SCALE
        )
        state_transitions[state_key][label] = result.get('transition', {})
        print(f'  → {len(state_transitions[state_key][label])} transition codes')

# Save to JSON
with open(GEE_EXPORTS_DIR / 'state_transitions.json', 'w') as f:
    json.dump(state_transitions, f, indent=2)
print('\n💾 Saved: state_transitions.json')

### 4.4: District-Level Transition Matrices

In [ ]:
# District-level transition matrices — export to Drive
for state_key, state_cfg in STATES.items():
    district_fc = boundaries[state_key]['district_fc']
    
    for year_t1, year_t2 in period_pairs:
        label = f'{year_t1}_{year_t2}'
        print(f'Computing district transitions: {state_cfg["name"]} {label}...')
        
        result_fc = compute_transition_matrix_districts(
            composites[state_key][year_t1],
            composites[state_key][year_t2],
            district_fc, scale=TIER1_SCALE
        )
        
        export_feature_collection_to_drive(
            result_fc,
            description=f'{state_key}_district_transitions_{label}',
            folder=GEE_DRIVE_FOLDER,
            file_format='CSV'
        )
        print(f'  → Export queued')

print('\n✅ All district transition exports queued.')

### 4.5: Water Body Vectorization (10m native)

In [ ]:
# Vectorize water bodies at 10m for accurate size classification
water_tasks = []

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    
    for year in YEARS:
        print(f'Vectorizing water bodies: {state_cfg["name"]} {year}...')
        
        water_fc = vectorize_water_bodies(
            composites[state_key][year], state_fc, scale=TIER1_SCALE
        )
        
        task = export_feature_collection_to_drive(
            water_fc,
            description=f'{state_key}_water_bodies_{year}',
            folder=GEE_DRIVE_FOLDER,
            file_format='GeoJSON'
        )
        water_tasks.append(task)

print(f'\n✅ {len(water_tasks)} water body exports queued.')

## Step 5: TIER 2 — Export 30m Full-State LULC Rasters

In [ ]:
# Export 30m compressed rasters for visualization
raster_tasks = []

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    state_geom = state_fc.geometry()
    
    for year in YEARS:
        desc = f'{state_key}_30m_{year}'
        print(f'Exporting raster: {desc}...')
        
        task = export_raster_to_drive(
            composites[state_key][year],
            description=desc,
            region=state_geom,
            scale=TIER2_SCALE,
            folder=GEE_DRIVE_FOLDER,
            crs=GEE_CRS
        )
        raster_tasks.append(task)

print(f'\n✅ {len(raster_tasks)} raster exports queued.')
print('   Expected sizes: Maharashtra ~80MB/file, Sikkim ~2MB/file')

## Step 6: Export DEM & Slope

In [ ]:
# Export SRTM DEM and slope for each state
dem_tasks = []

for state_key, state_cfg in STATES.items():
    state_fc = boundaries[state_key]['state_fc']
    state_geom = state_fc.geometry()
    
    srtm, slope = get_srtm_dem(state_fc)
    
    # Export elevation
    task_elev = export_raster_to_drive(
        srtm, f'{state_key}_elevation',
        region=state_geom, scale=30,
        folder=GEE_DRIVE_FOLDER
    )
    dem_tasks.append(task_elev)
    
    # Export slope
    task_slope = export_raster_to_drive(
        slope, f'{state_key}_slope',
        region=state_geom, scale=30,
        folder=GEE_DRIVE_FOLDER
    )
    dem_tasks.append(task_slope)

print(f'\n✅ {len(dem_tasks)} DEM exports queued.')

## Step 7: Download Administrative Boundaries

In [ ]:
# Export district boundaries from GEE as GeoJSON
for state_key, state_cfg in STATES.items():
    district_fc = boundaries[state_key]['district_fc']
    
    task = export_feature_collection_to_drive(
        district_fc,
        description=f'{state_key}_district_boundaries',
        folder=GEE_DRIVE_FOLDER,
        file_format='GeoJSON'
    )
    print(f'Exporting boundaries: {state_cfg["name"]}')

print('\n✅ Boundary exports queued.')
print('   After download, place files in data/raw/boundaries/')

## Step 8: Download OSM Road Network

This uses `osmnx` to query OpenStreetMap directly.

In [ ]:
import osmnx as ox

ROADS_DIR.mkdir(parents=True, exist_ok=True)

for state_key, state_cfg in STATES.items():
    state_name = state_cfg['name']
    print(f'Downloading road network: {state_name}...')
    print('  (This may take a few minutes for Maharashtra)')
    
    try:
        # Download drive network
        G = ox.graph_from_place(f"{state_name}, India", network_type='drive')
        roads = ox.graph_to_gdfs(G, nodes=False)
        
        # Filter to major roads
        # highway column can be list or string
        def is_major(highway):
            if isinstance(highway, list):
                return any(h in MAJOR_ROAD_TYPES for h in highway)
            return highway in MAJOR_ROAD_TYPES
        
        major_roads = roads[roads['highway'].apply(is_major)]
        
        # Save
        outpath = ROADS_DIR / f'{state_key}_major_roads.gpkg'
        major_roads.to_file(outpath, driver='GPKG')
        print(f'  ✅ Saved: {outpath} ({len(major_roads)} road segments)')
        
    except Exception as e:
        print(f'  ❌ Error: {e}')
        print(f'  → Try downloading from Geofabrik instead.')

print('\n✅ Road network download complete.')

## Step 9: Monitor GEE Export Tasks

In [ ]:
# Check all running GEE tasks
all_tasks = ee.batch.Task.list()
running = [t for t in all_tasks if t.status()['state'] in ('READY', 'RUNNING')]

print(f'Total tasks: {len(all_tasks)}')
print(f'Running/queued: {len(running)}')

for t in all_tasks[:20]:
    status = t.status()
    print(f"  {status['state']:10s} | {status['description']}")

In [ ]:
# Optionally: wait for all tasks to complete
# WARNING: This blocks. You can also check manually in GEE Code Editor.

# monitor_tasks(running, poll_interval=60)

## Step 10: Post-Download — Move Files

After all GEE exports complete and you've downloaded them from Google Drive:

1. Place 30m LULC GeoTIFFs in `data/raw/lulc/`
2. Place DEM/slope GeoTIFFs in `data/raw/dem/`
3. Place boundary GeoJSONs in `data/raw/boundaries/`
4. Place water body GeoJSONs in `data/gee_exports/`
5. Place composition/transition CSVs in `data/gee_exports/`

The CSV filenames should match:
- `maharashtra_district_composition_2016.csv`, etc.
- `maharashtra_district_transitions_2016_2020.csv`, etc.
- `maharashtra_water_bodies_2016.geojson`, etc.

In [ ]:
# Verify downloaded files
from pathlib import Path

print('=== Data Directory Status ===')
for subdir in [LULC_DIR, DEM_DIR, BOUNDARIES_DIR, ROADS_DIR, GEE_EXPORTS_DIR]:
    files = list(subdir.glob('*')) if subdir.exists() else []
    print(f'\n{subdir.relative_to(Path(".."))}/ ({len(files)} files)')
    for f in files:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name} ({size_mb:.1f} MB)')

---

## Summary

| Export Type | Description | Count | Est. Size |
|---|---|---|---|
| State compositions | JSON (pixel counts) | 1 file | ~5 KB |
| District compositions | CSVs from Drive | 6 files | ~1 MB total |
| State transitions | JSON (transition codes) | 1 file | ~50 KB |
| District transitions | CSVs from Drive | 6 files | ~5 MB total |
| Water body vectors | GeoJSON from Drive | 6 files | ~50 MB total |
| LULC rasters (30m) | GeoTIFF from Drive | 6 files | ~250 MB total |
| DEM + slope | GeoTIFF from Drive | 4 files | ~100 MB total |
| District boundaries | GeoJSON from Drive | 2 files | ~10 MB total |
| Road network | GeoPackage (local) | 2 files | ~50 MB total |
| **Total** | | | **~500 MB** |

**Next**: Proceed to `02_lulc_composition.ipynb` to analyze compositions and build transition matrices.